# Imports

In [1]:
import re
import optuna
import pickle

import numpy as np
import pandas as pd

from sklearn import set_config
from lightgbm import LGBMClassifier

from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict

from feature_engine.selection import DropFeatures

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
set_config(enable_metadata_routing=True)

# Loading Dataset

In [3]:
X_train = pd.read_parquet('../data/processed/X_train_logit.parquet')
y_train = pd.read_parquet('../data/processed/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_logit.parquet')

In [4]:
X_train.head()

,lgbm_logit,cat_logit,xgb_logit,hist_logit,lg_logit,knn_logit,ridge_logit,voting_lgbm_cat_xgb_hist_logit,voting_lg_ridge_logit,stacking_lgbm_with_knn_and_voting_lg_ridge_and_voting_lgbm_cat_xgb_hist_logit
0,1.402666,2.109990,2.149899,2.030346,1.440273,2.064999,0.500714,1.887848,0.925326,1.226844
1,0.616425,1.595299,1.584592,1.262993,0.799597,0.079907,0.335105,1.222563,0.561368,0.371762
2,0.023615,1.748857,1.106745,1.146720,0.235157,-1.325798,0.136991,0.929687,0.186260,0.066618
3,-6.367775,-5.125195,-4.946636,-4.912857,-6.083520,-34.538776,-1.247050,-5.198584,-2.069743,-6.799934
4,-0.438053,0.263538,-0.043875,-0.068158,-0.346862,-0.861416,-0.094611,-0.070366,-0.220629,-0.878595


In [5]:
X_test.head()

,lgbm_logit,cat_logit,xgb_logit,hist_logit,lg_logit,knn_logit,ridge_logit,voting_lgbm_cat_xgb_hist_logit,voting_lg_ridge_logit,stacking_lgbm_with_knn_and_voting_lg_ridge_and_voting_lgbm_cat_xgb_hist_logit
0,-5.379032,-3.409957,-3.759100,-3.888929,-5.042109,-34.538776,-1.272709,-3.899528,-34.538776,-5.055490
1,-5.795381,-4.426299,-4.187473,-3.950213,-2.684867,-2.019087,-0.626960,-4.396275,-34.538776,-5.602505
2,-5.574937,-4.441487,-4.232580,-4.454119,-5.395432,-34.538776,-1.335019,-4.564839,-34.538776,-5.656104
3,-1.635113,0.176618,0.347033,-0.058463,0.644945,-2.198160,0.190133,-0.222683,-0.304675,-1.163384
4,1.920299,3.398116,3.163860,3.161400,1.661579,34.539576,0.535153,2.745303,0.794179,2.146481


# Machine Learning

In [6]:
def objective(trial, X, y):

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    aucs = []

    params = {
        "solver": trial.suggest_categorical("solver", ["saga"]),
        "C": trial.suggest_float("C", 1e-5, 100, log=True),
        "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0),
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
        "fit_intercept": trial.suggest_categorical("fit_intercept", [True, False]),
        "tol": trial.suggest_float("tol", 1e-6, 1e-2, log=True),
        "max_iter": trial.suggest_int("max_iter", 1000, 5000)
    }

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx, 0], y.iloc[valid_idx, 0]

        model = LogisticRegression(**params)
        model.fit(X_train, y_train)

        proba = model.predict_proba(X_valid)[:, 1]

        auc = roc_auc_score(y_valid, proba)
        aucs.append(auc)

        trial.report(np.mean(aucs), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(aucs)

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=500, n_jobs=-1, show_progress_bar=True)


print("Best trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-05-19 15:36:52,023] A new study created in memory with name: no-name-e28d4f89-23ff-45a0-ad5e-6fa68c9499e5
Best trial: 9. Best value: 0.944693:   0%|▏                                                                                                               | 1/500 [00:10<1:30:55, 10.93s/it]

[I 2026-05-19 15:37:02,933] Trial 9 finished with value: 0.9446932299701961 and parameters: {'solver': 'saga', 'C': 2.06198041382091e-05, 'l1_ratio': 0.9817538850496152, 'class_weight': None, 'fit_intercept': False, 'tol': 0.0007496390764380842, 'max_iter': 3982}. Best is trial 9 with value: 0.9446932299701961.


Best trial: 1. Best value: 0.949534:   0%|▍                                                                                                               | 2/500 [00:16<1:05:01,  7.83s/it]

[I 2026-05-19 15:37:08,617] Trial 1 finished with value: 0.9495342489125745 and parameters: {'solver': 'saga', 'C': 0.00032532931605422073, 'l1_ratio': 0.894567137202324, 'class_weight': 'balanced', 'fit_intercept': False, 'tol': 9.520621465954567e-05, 'max_iter': 3304}. Best is trial 1 with value: 0.9495342489125745.


Best trial: 4. Best value: 0.949628:   1%|▋                                                                                                                 | 3/500 [00:17<39:14,  4.74s/it]

...

[I 2026-05-19 17:00:01,467] Trial 497 finished with value: 0.9499120594902412 and parameters: {'solver': 'saga', 'C': 0.025709150356692054, 'l1_ratio': 0.4404217231682701, 'class_weight': None, 'fit_intercept': True, 'tol': 9.57135554902691e-06, 'max_iter': 3846}. Best is trial 250 with value: 0.9499121585509325.


Best trial: 250. Best value: 0.949912: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████▊| 499/500 [1:23:12<00:07,  7.71s/it]

[I 2026-05-19 17:00:04,967] Trial 498 finished with value: 0.949911934561561 and parameters: {'solver': 'saga', 'C': 0.024203978326368898, 'l1_ratio': 0.44399438486325166, 'class_weight': None, 'fit_intercept': True, 'tol': 8.88685687393311e-06, 'max_iter': 4246}. Best is trial 250 with value: 0.9499121585509325.


Best trial: 250. Best value: 0.949912: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [1:23:22<00:00, 10.01s/it]

[I 2026-05-19 17:00:14,912] Trial 499 finished with value: 0.9499118877162429 and parameters: {'solver': 'saga', 'C': 0.022772733984773322, 'l1_ratio': 0.44251844076905555, 'class_weight': None, 'fit_intercept': True, 'tol': 9.468615886648325e-06, 'max_iter': 4279}. Best is trial 250 with value: 0.9499121585509325.
Best trial score:
0.9499121585509325

Best params:
{'solver': 'saga', 'C': 0.02914879956934508, 'l1_ratio': 0.43970786556822744, 'class_weight': None, 'fit_intercept': True, 'tol': 8.83661241539477e-06, 'max_iter': 3921}


In [8]:
stacking_lg = LogisticRegression(**study.best_params).fit(X_train, y_train.PitNextLap)

# Submission

In [9]:
submission = pd.read_csv('../data/raw/sample_submission.csv')

In [10]:
submission['PitNextLap'] = stacking_lg.predict_proba(X_test)[:, 1]

In [11]:
submission.to_csv('../data/submission/submission.csv', index=False)

In [12]:
X_train.columns

Index(['lgbm_logit', 'cat_logit', 'xgb_logit', 'hist_logit', 'lg_logit',
       'knn_logit', 'ridge_logit', 'voting_lgbm_cat_xgb_hist_logit',
       'voting_lg_ridge_logit',
       'stacking_lgbm_with_knn_and_voting_lg_ridge_and_voting_lgbm_cat_xgb_hist_logit'],
      dtype='str')